In [1]:
from dotenv import load_dotenv
import os
import pandas as pd
import anthropic
import mlflow
from mlflow.metrics import latency
from mlflow.metrics.genai import faithfulness, answer_correctness

In [2]:
# Load environment files and store API keys
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_KEY")

In [3]:
# Read in dialogue data
file_path = "../../data/Hallucination/dialogue_data.json"
dialogue_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/general_data.json"
general_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/qa_data.json"
qa_data = pd.read_json(file_path, lines=True)

In [4]:
class Claude():
    """Class to implement Claude model for DeepEval"""
    def __init__(self, model, api_key):
        self.model = model
        self.api_key = api_key
        self.client = anthropic.Client(api_key=self.api_key)

    def load_model(self):
        return self.model

    def generate(self, prompt, max_tokens = 4096):
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return response.content[0].text

    async def a_generate(self, prompt, max_tokens = 4096):
        
        model = self.load_model()
        
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return response.content[0].text

    def get_model_name(self):
        return "Claude Model"

In [5]:
# Define Claude model
claude_model = "claude-3-5-haiku-20241022"

# Create Claude model
claude_instance = Claude(model=claude_model, api_key=anthropic_key)

# Test functionality
print(claude_instance.generate("Hello, Claude"))

Hello! How are you doing today?


In [ ]:
# Function to query the LLM with hallucination-aware prompt
def evaluate_hallucination(context, question):

In [6]:
eval_data = pd.DataFrame({
    'inputs': qa_data.question,
    'predictions': qa_data.hallucinated_answer,
    'ground_truth': qa_data.knowledge
})

eval_data = eval_data.head()

In [8]:
from mlflow.metrics.genai import EvaluationExample, faithfulness

# Create a good and bad example for faithfulness in the context of this problem
faithfulness_examples = [
    EvaluationExample(
        input="How do I disable MLflow autologging?",
        output="mlflow.autolog(disable=True) will disable autologging for all functions. In Databricks, autologging is enabled by default. ",
        score=2,
        justification="The output provides a working solution, using the mlflow.autolog() function that is provided in the context.",
        grading_context={
            "context": "mlflow.autolog(log_input_examples: bool = False, log_model_signatures: bool = True, log_models: bool = True, log_datasets: bool = True, disable: bool = False, exclusive: bool = False, disable_for_unsupported_versions: bool = False, silent: bool = False, extra_tags: Optional[Dict[str, str]] = None) → None[source] Enables (or disables) and configures autologging for all supported integrations. The parameters are passed to any autologging integrations that support them. See the tracking docs for a list of supported autologging integrations. Note that framework-specific configurations set at any point will take precedence over any configurations set by this function."
        },
    ),
    EvaluationExample(
        input="How do I disable MLflow autologging?",
        output="mlflow.autolog(disable=True) will disable autologging for all functions.",
        score=5,
        justification="The output provides a solution that is using the mlflow.autolog() function that is provided in the context.",
        grading_context={
            "context": "mlflow.autolog(log_input_examples: bool = False, log_model_signatures: bool = True, log_models: bool = True, log_datasets: bool = True, disable: bool = False, exclusive: bool = False, disable_for_unsupported_versions: bool = False, silent: bool = False, extra_tags: Optional[Dict[str, str]] = None) → None[source] Enables (or disables) and configures autologging for all supported integrations. The parameters are passed to any autologging integrations that support them. See the tracking docs for a list of supported autologging integrations. Note that framework-specific configurations set at any point will take precedence over any configurations set by this function."
        },
    ),
]

faithfulness_metric = faithfulness(model="openai:/gpt-4", examples=faithfulness_examples)

In [9]:
eval_df = pd.DataFrame(
    {
        "questions": [
            "What is MLflow?",
            "How to run mlflow.evaluate()?",
            "How to log_table()?",
            "How to load_table()?",
        ],
        "context": [
            "It's an apple",
            "You put a pin in it",
            "I don't know, try",
            "This is Medford, NY"
        ],
        "results": [
            "It's an apple",
            "You put a pin in it",
            "I don't know, try",
            "This is Medford, NY"
        ]
    }
)

In [10]:
results = mlflow.evaluate(
    data=eval_df,
    #model_type="question-answering",
    #evaluators="default",
    predictions="result",
    extra_metrics=[faithfulness_metric],
    evaluator_config={
        "col_mapping": {
            "inputs": "questions",
            "context": "context",
            "result": "results"
        }
    },
)
print(results.metrics)

2025/04/12 22:30:30 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
c:\Users\Johnh\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1/1 [00:01<00:00,  1.35s/it]
c:\Users\Johnh\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Johnh\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Johnh\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:4268: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, a

{'faithfulness/v1/mean': np.float64(nan), 'faithfulness/v1/variance': np.float64(nan)}


In [38]:
from mlflow.metrics.genai import EvaluationExample, answer_similarity

# Create an example to describe what answer_similarity means like for this problem.
example = EvaluationExample(
  input="What is MLflow?",
  output="MLflow is an open-source platform for managing machine "
  "learning workflows, including experiment tracking, model packaging, "
  "versioning, and deployment, simplifying the ML lifecycle.",
  score=4,
  justification="The definition effectively explains what MLflow is "
  "its purpose, and its developer. It could be more concise for a 5-score.",
  grading_context={
      "targets": "MLflow is an open-source platform for managing "
      "the end-to-end machine learning (ML) lifecycle. It was developed by Databricks, "
      "a company that specializes in big data and machine learning solutions. MLflow is "
      "designed to address the challenges that data scientists and machine learning "
      "engineers face when developing, training, and deploying machine learning models."
  },
)

# Construct the metric using Claude as the judge
answer_similarity_metric = answer_similarity(model=f"anthropic:/{claude_model}", examples=[example])

In [55]:
import mlflow
from mlflow.metrics.genai import faithfulness
import pandas as pd

eval_data = pd.DataFrame({
    "inputs": [
        "What is the capital of France?",
        "Who discovered penicillin?"
    ],
    "prediction": [
        "The capital of France is Paris.",
        "Penicillin was discovered by Alexander Fleming in 1928."
    ],
    "ground_truth": [
        "Paris",
        "Alexander Fleming"
    ],
    "context": [
        "France's capital is Paris. It is known for the Eiffel Tower.",
        "Alexander Fleming discovered penicillin in 1928."
    ]
})


# Define Claude model as the judge — this assumes Claude is accessible via proxy or model URI
# For OpenAI, it would be: model="openai:/gpt-4"
results = faithfulness(
    model=f"anthropic:/{claude_model}",  # <-- This needs to be a valid URI for MLflow
    metric_version="v1",
    examples=[],  # Optional: can include example comparisons
    parameters={"temperature": 0.0},
    max_workers=5,
    metric_metadata=None,
    proxy_url=None,
    extra_headers=None
)


In [56]:
results = mlflow.evaluate(
    model=None,  # placeholder model type
    data=eval_data,
    targets="ground_truth",
    predictions="prediction",
    model_type="question-answering",
    extra_metrics=[
        faithfulness()  # Use your judge model here
    ],
    evaluator_config={
        "col_mapping": {
            "input": "question",
            "context": "context"
        }
    }
)

2025/04/12 23:14:50 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
Using default facebook/roberta-hate-speech-dynabench-r4-target checkpoint
2025/04/12 23:14:50 WARNING mlflow.metrics.metric_definitions: Failed to load 'toxicity' metric (error: RuntimeError('At least one of TensorFlow 2.0 or PyTorch should be installed. To install TensorFlow 2.0, read the instructions at https://www.tensorflow.org/install/ To install PyTorch, read the instructions at https://pytorch.org/.')), skipping metric logging.
2025/04/12 23:14:50 WARNING mlflow.models.evaluation.utils.metric: Did not log metric 'toxicity' at index 1 in the `extra_metrics` parameter because it returned None.
100%|██████████| 1/1 [00:00<00:00, 3715.06it/s]
c:\Users\Johnh\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Johnh\AppData\Local\Programs\Python\Python

In [57]:
results.metrics

{'flesch_kincaid_grade_level/v1/mean': np.float64(8.4),
 'flesch_kincaid_grade_level/v1/variance': np.float64(15.210000000000003),
 'flesch_kincaid_grade_level/v1/p90': np.float64(11.520000000000001),
 'ari_grade_level/v1/mean': np.float64(6.4),
 'ari_grade_level/v1/variance': np.float64(19.360000000000003),
 'ari_grade_level/v1/p90': np.float64(9.920000000000002),
 'exact_match/v1': 0.0,
 'faithfulness/v1/mean': np.float64(nan),
 'faithfulness/v1/variance': np.float64(nan)}